<!-- open-in-badges -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/05_sequences/notebooks/sesion_05_4_texto_embeddings.ipynb) [![Abrir en Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/05_sequences/notebooks/sesion_05_4_texto_embeddings.ipynb) [![Abrir en Lightning Studio](https://pl-bolts-doc-images.s3.us-east-2.amazonaws.com/app-2/studio-badge.svg)](https://lightning.ai/new?repo_url=https%3A%2F%2Fgithub.com%2FEAFIT-IA%2Fsi7011-DeepLearning%2Fblob%2Fmain%2Fsessions%2F05_sequences%2Fnotebooks%2Fsesion_05_4_texto_embeddings.ipynb)

# SI7011 · Sesión 05 · 4. Texto como secuencia: tokens, embeddings y orden
**Juan David Martínez Vargas · Universidad EAFIT**

Acompaña la **Parte 4** de las diapositivas. Reseñas de películas de IMDB, positivas o negativas. Tres clasificadores con el mismo embedding y el mismo presupuesto, que se diferencian solo en **cuánto del orden** de las palabras ven.

| Sección | Lo que cambia | Qué mirar |
|---|---|---|
| 1 | texto → tokens → vocabulario → ids | el vocabulario sale solo de train |
| 2 | lotes con *padding* y máscara | el error silencioso de promediar el padding |
| 3 | *bag of embeddings*: sin orden | la referencia |
| 4 | CNN 1-D: orden local, 3 palabras | ¿le gana al promedio? |
| 5 | LSTM: todo el orden | ¿y esta? |
| 6 | barajar las palabras de cada reseña de prueba | ¿cuánto del orden usa cada modelo? |
| 7 | negaciones a mano | "good" frente a "not good" |
| 8 | vecinos en el embedding $E$ | qué aprendió $E$ |
| 9 | $E$ preentrenado (GloVe), congelado o ajustado | ¿cuánto vale traer el embedding de otro texto? |

**Datos:** IMDB (Maas et al., 2011): 25 000 reseñas de entrenamiento y 25 000 de prueba, mitad y mitad. **Usa GPU** (Colab o Kaggle): unos 3 minutos. En CPU corre, pero la LSTM tarda unos 5 minutos.

In [1]:
import json, time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn

torch.manual_seed(0)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
print('PyTorch', torch.__version__)

import re, collections
import pandas as pd
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

PyTorch 2.14.1+cu130


device: cpu


### Cargar IMDB
La celda busca los archivos `train-00000-of-00001.parquet` y `test-00000-of-00001.parquet` de [`stanfordnlp/imdb`](https://huggingface.co/datasets/stanfordnlp/imdb) en este orden y usa el primero que encuentre. Imprime la **fuente** usada: revísala.

| Plataforma | De dónde salen los datos |
|---|---|
| **Lightning Studio** o **local** | la carpeta `data/` junto al notebook (por ejemplo `data/imdb/`) |
| **Kaggle** o **Colab** | se descargan de Hugging Face con `datasets` (en Kaggle, deja *Internet* encendido) |

In [2]:
from pathlib import Path

def buscar(nombre):
    for root in [Path('data'), Path('/kaggle/input'), Path('/teamspace/studios/this_studio/data')]:
        if root.exists():
            hits = sorted(root.rglob(nombre))
            if hits:
                return hits[0]
    return None

archivos = [buscar(f'{s}-00000-of-00001.parquet') for s in ('train', 'test')]
if all(archivos):
    train_df, test_df = (pd.read_parquet(a) for a in archivos)
    fuente = str(archivos[0].parent)
else:
    from datasets import load_dataset
    imdb = load_dataset('stanfordnlp/imdb')
    train_df, test_df = imdb['train'].to_pandas(), imdb['test'].to_pandas()
    fuente = 'Hugging Face stanfordnlp/imdb'
print('fuente:', fuente)
print(f'train {len(train_df):,} · test {len(test_df):,} · positivas en train {train_df.label.mean():.0%}')
print(train_df.text.iloc[3][:400])

fuente: data/imdb
train 25,000 · test 25,000 · positivas en train 50%
This film was probably inspired by Godard's Masculin, féminin and I urge you to see that film instead.<br /><br />The film has two strong elements and those are, (1) the realistic acting (2) the impressive, undeservedly good, photo. Apart from that, what strikes me most is the endless stream of silliness. Lena Nyman has to be most annoying actress in the world. She acts so stupid and with all the 


## 1. De texto a enteros
**Tokenizar** con una regla simple: minúsculas, palabras y algunos signos, sin los `<br />` del HTML. **Vocabulario:** las 20 000 palabras más frecuentes de las reseñas de **entrenamiento**; todo lo demás va a `[UNK]`, y `[PAD]` rellena. Separamos 5 000 reseñas de train para validar **antes** de construir el vocabulario: la validación no puede decidir qué palabras existen.

In [3]:
def tokenizar(s):
    return re.findall(r"[a-z0-9']+|[!?.,]", s.lower().replace('<br />', ' '))

tok_train = [tokenizar(s) for s in train_df.text]
tok_test = [tokenizar(s) for s in test_df.text]
y_train = torch.tensor(train_df.label.values)
y_test = torch.tensor(test_df.label.values)

orden = np.random.default_rng(0).permutation(len(tok_train))
i_tr, i_va = orden[:20000], orden[20000:]                       # 20 000 para entrenar, 5 000 para validar

conteo = collections.Counter(w for i in i_tr for w in tok_train[i])
VOCAB = ['[PAD]', '[UNK]'] + [w for w, _ in conteo.most_common(20000)]
W2I = {w: i for i, w in enumerate(VOCAB)}
T_MAX = 256                                                     # truncar: las primeras 256 palabras

def a_ids(tokens):
    return [W2I.get(w, 1) for w in tokens][:T_MAX]

ids_train = [a_ids(t) for t in tok_train]
ids_test = [a_ids(t) for t in tok_test]
largos = np.array([len(t) for t in tok_train])
print(f'{len(VOCAB):,} tokens en el vocabulario · mediana {np.median(largos):.0f} palabras por reseña · '
      f'{(largos > T_MAX).mean():.0%} se truncan a {T_MAX}')
print(tok_train[3][:12], '→', ids_train[3][:12])
print(f'tokens [UNK] en prueba: {np.mean([i == 1 for s in ids_test for i in s]):.1%}')

20,002 tokens en el vocabulario · mediana 194 palabras por reseña · 34% se truncan a 256
['this', 'film', 'was', 'probably', 'inspired', 'by', "godard's", 'masculin', ',', 'f', 'minin', 'and'] → [13, 21, 15, 243, 1543, 34, 1, 1, 4, 1196, 1, 5]


tokens [UNK] en prueba: 3.0%


## 2. Lotes de distinto largo: *padding* y máscara
Cada lote se rellena con `[PAD]` (id 0) hasta la reseña más larga del lote, y la máscara dice qué posiciones son reales. Todos los modelos deben ignorar el padding.

In [4]:
def lote(secuencias):
    largo = torch.tensor([max(len(s), 1) for s in secuencias])
    X = torch.zeros(len(secuencias), int(largo.max()), dtype=torch.long)         # 0 = [PAD]
    for i, s in enumerate(secuencias):
        X[i, :len(s)] = torch.tensor(s)
    return X, largo

X, largo = lote([ids_train[i] for i in range(4)])
print('lote', tuple(X.shape), '· largos', largo.tolist())
print('máscara de la reseña más corta:', (X[int(largo.argmin())] != 0).int()[:largo.max()].tolist()[-12:], '…')

lote (4, 256) · largos [256, 239, 95, 140]
máscara de la reseña más corta: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] …


In [5]:
def exactitud(modelo, ids, y, barajar=False, semilla=1):
    """Exactitud; con barajar=True se permutan las palabras de cada reseña (el modelo es el mismo)."""
    modelo.eval(); g = np.random.default_rng(semilla); ok = 0
    with torch.no_grad():
        for i in range(0, len(ids), 500):
            sec = [list(g.permutation(s)) if barajar else s for s in ids[i:i + 500]]
            X, largo = lote(sec)
            ok += ((modelo(X.to(device), largo) > 0).cpu().long() == y[i:i + 500]).sum().item()
    return ok / len(ids)

def entrenar_texto(modelo, epocas=3, lr=2e-3, tam=64, indices=None):
    modelo.to(device); t0 = time.time()
    entrenables = [q for q in modelo.parameters() if q.requires_grad]      # un embedding congelado no entra
    opt = torch.optim.Adam(entrenables, lr)
    va_ids, va_y = [ids_train[j] for j in i_va], y_train[i_va]
    for ep in range(epocas):
        modelo.train()
        p = np.random.default_rng(ep).permutation(i_tr if indices is None else indices)
        for i in range(0, len(p), tam):
            b = p[i:i + tam]
            X, largo = lote([ids_train[j] for j in b])
            # entropía cruzada binaria de la sesión 01, desde logits
            loss = nn.functional.binary_cross_entropy_with_logits(modelo(X.to(device), largo), y_train[b].float().to(device))
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(modelo.parameters(), 1.0); opt.step()
        print(f'época {ep + 1}: exactitud de validación {exactitud(modelo, va_ids, va_y):.3f}')
    print(f'{time.time() - t0:.0f} s · {sum(q.numel() for q in modelo.parameters()):,} parámetros, '
          f'{sum(q.numel() for q in entrenables):,} entrenables')
    return modelo

resultados = {}
def anotar(nombre, modelo):
    resultados[nombre] = {'test': exactitud(modelo, ids_test, y_test),
                          'test_barajado': exactitud(modelo, ids_test, y_test, barajar=True),
                          'parametros': sum(p.numel() for p in modelo.parameters())}
    r = resultados[nombre]
    print(f"{nombre}: prueba {r['test']:.3f} · prueba con palabras barajadas {r['test_barajado']:.3f}")

## 3. *Bag of embeddings*: el promedio, sin orden
$\frac{1}{T}\sum_t \mathbf e_{w_t}$ y una capa lineal. La máscara hace que el promedio sea solo sobre palabras reales: si se promediara también el padding, las reseñas cortas quedarían arrastradas hacia el vector de `[PAD]`.

In [6]:
class BolsaEmbeddings(nn.Module):
    def __init__(self, d=64):
        super().__init__()
        self.E = nn.Embedding(len(VOCAB), d, padding_idx=0)          # e_w = E[w]
        self.salida = nn.Linear(d, 1)
    def forward(self, X, largo):
        m = (X != 0).float().unsqueeze(-1)                            # la máscara: 1 en palabras reales
        promedio = (self.E(X) * m).sum(1) / m.sum(1).clamp(min=1)     # promedio SOLO sobre palabras reales
        return self.salida(promedio).squeeze(-1)

torch.manual_seed(0)
bolsa = entrenar_texto(BolsaEmbeddings())
anotar('bolsa', bolsa)

época 1: exactitud de validación 0.804


época 2: exactitud de validación 0.854


época 3: exactitud de validación 0.862
10 s · 1,280,193 parámetros, 1,280,193 entrenables


bolsa: prueba 0.854 · prueba con palabras barajadas 0.854


## 4. CNN 1-D: orden dentro de 3 palabras
64 filtros sobre ventanas de 3 palabras y el máximo sobre el tiempo: cada filtro detecta un patrón local ("not very good") en cualquier lugar de la reseña.

In [7]:
class CNNTexto(nn.Module):
    def __init__(self, d=64, filtros=64):
        super().__init__()
        self.E = nn.Embedding(len(VOCAB), d, padding_idx=0)
        self.conv = nn.Conv1d(d, filtros, kernel_size=3, padding=1)
        self.salida = nn.Linear(filtros, 1)
    def forward(self, X, largo):
        h = torch.relu(self.conv(self.E(X).transpose(1, 2)))         # (B, filtros, T)
        h = h.masked_fill((X == 0).unsqueeze(1), -1e9)                # el padding no puede ganar el máximo
        return self.salida(h.max(dim=2).values).squeeze(-1)

torch.manual_seed(0)
cnn = entrenar_texto(CNNTexto())
anotar('cnn', cnn)

época 1: exactitud de validación 0.807


época 2: exactitud de validación 0.842


época 3: exactitud de validación 0.843
20 s · 1,292,545 parámetros, 1,292,545 entrenables


cnn: prueba 0.841 · prueba con palabras barajadas 0.796


## 5. LSTM: todo el orden
La LSTM de la Parte 3 sobre los embeddings. `pack_padded_sequence` hace que cada reseña termine en su última palabra real: el estado $\mathbf h_T$ no pasa por el padding.

In [8]:
class LSTMTexto(nn.Module):
    def __init__(self, d=64, oculta=64):
        super().__init__()
        self.E = nn.Embedding(len(VOCAB), d, padding_idx=0)
        self.lstm = nn.LSTM(d, oculta, batch_first=True)
        self.salida = nn.Linear(oculta, 1)
    def forward(self, X, largo):
        empacado = nn.utils.rnn.pack_padded_sequence(self.E(X), largo.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_T, _) = self.lstm(empacado)                             # h_T de la última palabra REAL de cada reseña
        return self.salida(h_T[-1]).squeeze(-1)

torch.manual_seed(0)
lstm = entrenar_texto(LSTMTexto())
anotar('lstm', lstm)

época 1: exactitud de validación 0.712


época 2: exactitud de validación 0.769


época 3: exactitud de validación 0.828
340 s · 1,313,473 parámetros, 1,313,473 entrenables


lstm: prueba 0.826 · prueba con palabras barajadas 0.807


## 6. Barajar las palabras
Los tres modelos ya están entrenados con texto normal. Ahora se permutan al azar las palabras de cada reseña de prueba y se mide otra vez.

In [9]:
nombres = {'bolsa': 'bag of embeddings', 'cnn': 'CNN 1-D', 'lstm': 'LSTM'}
print(f"{'modelo':18s} {'parámetros':>10s} {'prueba':>7s} {'barajada':>9s}")
for k, n in nombres.items():
    r = resultados[k]
    print(f"{n:18s} {r['parametros']:10,d} {r['test']:7.3f} {r['test_barajado']:9.3f}")

modelo             parámetros  prueba  barajada
bag of embeddings   1,280,193   0.854     0.854
CNN 1-D             1,292,545   0.841     0.796
LSTM                1,313,473   0.826     0.807


El promedio da exactamente lo mismo con las palabras barajadas: su fórmula no mira el orden. La CNN y la LSTM **sí** pierden al barajar, así que usan el orden. Pero no le ganan al promedio con el texto normal: para el sentimiento de una reseña larga, qué palabras aparecen dice casi todo, y el orden aporta poco que compense un modelo más difícil de entrenar.

La LSTM todavía estaba mejorando en la última época; con más épocas se acerca a las otras dos, sin superarlas claramente (*Tu turno*).

## 7. Negaciones, a mano
Frases cortas inventadas, sin truncar ni barajar. La salida es la probabilidad de "positiva", $\sigma(\text{logit})$.

In [10]:
frases = ['this movie was good', 'this movie was not good', 'this movie was bad', 'this movie was not bad',
          'i expected it to be terrible but it was great', 'i expected it to be great but it was terrible']
with torch.no_grad():
    for f in frases:
        X, largo = lote([a_ids(tokenizar(f))])
        p = {k: torch.sigmoid(m.eval()(X.to(device), largo)).item() for k, m in (('bolsa', bolsa), ('cnn', cnn), ('lstm', lstm))}
        print(f"{f:48s} bolsa {p['bolsa']:.2f} · CNN {p['cnn']:.2f} · LSTM {p['lstm']:.2f}")

this movie was good                              bolsa 0.51 · CNN 0.13 · LSTM 0.43
this movie was not good                          bolsa 0.01 · CNN 0.04 · LSTM 0.35
this movie was bad                               bolsa 0.00 · CNN 0.00 · LSTM 0.18
this movie was not bad                           bolsa 0.00 · CNN 0.00 · LSTM 0.19
i expected it to be terrible but it was great    bolsa 0.62 · CNN 0.90 · LSTM 0.38
i expected it to be great but it was terrible    bolsa 0.62 · CNN 0.17 · LSTM 0.21


Mira las dos últimas frases: tienen **las mismas palabras** en otro orden, así que el promedio está obligado a darles la misma probabilidad. Los modelos con orden pueden distinguirlas, y la CNN lo hace en el sentido correcto: "but it was great" sube, "but it was terrible" baja.

Y mira "not bad": para el promedio, *not* es una palabra negativa más (aparece mucho en reseñas negativas), así que "not bad" queda tan negativa como "bad". Son seis frases, no una evaluación: sirven para preguntar, no para concluir.

## 8. Qué aprendió el embedding
Vecinos por similitud coseno en la matriz $E$ del promedio (sección 3), y el puntaje de cada palabra en la capa de salida, $\mathbf w^\top \mathbf e_w$: positivo empuja hacia "positiva".

In [11]:
E = bolsa.E.weight.detach().cpu()
En = nn.functional.normalize(E, dim=1)
w = bolsa.salida.weight.detach().cpu()[0]
puntaje = lambda i: float(E[i] @ w)

CONSULTA = ['excellent', 'terrible', 'boring', 'beautiful', 'waste', 'actor']
vecinos = []
for palabra in CONSULTA:
    i = W2I[palabra]
    sim = En @ En[i]
    sim[[0, 1, i]] = -1                                             # sin [PAD], [UNK] ni la propia palabra
    top = sim.topk(5).indices.tolist()
    vecinos.append({'palabra': palabra, 'puntaje': puntaje(i),
                    'vecinos': [[VOCAB[j], round(float(sim[j]), 2)] for j in top],
                    'signo': {VOCAB[j]: (1 if puntaje(j) > 0 else -1) for j in top}})
    print(f"{palabra:10s} ({puntaje(i):+.2f}) →", ', '.join(f'{VOCAB[j]} ({puntaje(j):+.1f})' for j in top))

excellent  (+66.87) → simple (+44.2), loved (+54.3), great (+63.8), best (+53.5), superb (+57.4)
terrible   (-58.05) → worse (-73.7), worst (-101.1), holiday (-30.0), stupid (-57.1), embarrassing (-50.6)
boring     (-74.61) → worst (-101.1), ridiculous (-60.1), transparent (-29.4), disappointment (-55.9), fails (-52.3)
beautiful  (+40.48) → great (+63.8), complex (+35.1), favorite (+49.9), captures (+48.5), tears (+32.3)
waste      (-70.95) → worse (-73.7), worst (-101.1), awful (-71.8), poorly (-59.9), bad (-71.5)
actor      (-25.27) → resurrection (-19.0), wrong (-30.6), been (-28.3), leonard's (-16.0), sirk (+7.0)


Los vecinos se agrupan por **polaridad**, no por significado: el embedding se entrenó para una sola tarea, separar reseñas positivas de negativas, y eso es lo que organiza el espacio. Un embedding es útil **para su tarea** (sesión 04). Una palabra sin opinión propia, como "actor", queda con un puntaje más moderado que las de opinión y con vecinos que no se parecen en nada: el embedding no aprendió qué es un actor, solo hacia qué lado empuja.

In [12]:
# Lo que lee la figura S05-F07 (make_s05_figuras_notebooks.py)
json.dump(vecinos, open('vecinos_embedding.json', 'w'), ensure_ascii=False, indent=1)

## 9. Embeddings preentrenados: GloVe
Hasta aquí $E$ empezó aleatorio y lo aprendió la tarea, con 20 000 reseñas y una sola señal: positiva o negativa. GloVe (Pennington et al., 2014) es un $E$ aprendido **antes**, sin etiquetas, de unos 6 000 millones de palabras de Wikipedia y noticias, a partir de qué palabras aparecen cerca de cuáles. Probamos dos formas de usarlo en el mismo promedio de la sección 3:

- **congelado:** $E$ = GloVe y no se toca; solo se entrena la capa lineal (101 parámetros);
- **ajustado:** $E$ empieza en GloVe y se sigue entrenando con la tarea.

Es la primera vez en la sesión que usamos algo preentrenado; la sesión 06 trata esto a fondo.

In [13]:
import gzip, urllib.request
GLOVE = 'glove-wiki-gigaword-100.gz'           # GloVe 6B, 400 000 palabras, d = 100 (paquete de gensim-data)
URL = f'https://github.com/RaRe-Technologies/gensim-data/releases/download/glove-wiki-gigaword-100/{GLOVE}'
ruta_glove = buscar(GLOVE)
if ruta_glove is None:
    ruta_glove = Path('data/glove') / GLOVE
    ruta_glove.parent.mkdir(parents=True, exist_ok=True)
    print('descargando GloVe (130 MB)…'); urllib.request.urlretrieve(URL, ruta_glove)
print('GloVe:', ruta_glove)

D_G = 100
torch.manual_seed(0)
E0 = torch.randn(len(VOCAB), D_G) * 0.1        # palabras que GloVe no tiene: vector pequeño al azar
E0[0] = 0                                       # [PAD]
encontradas = 0
with gzip.open(ruta_glove, 'rt', encoding='utf-8') as f:
    next(f)                                     # primera línea: '400000 100'
    for linea in f:
        palabra, *v = linea.rstrip().split(' ')
        if palabra in W2I:
            E0[W2I[palabra]] = torch.tensor([float(x) for x in v]); encontradas += 1
print(f'{encontradas:,} de {len(VOCAB):,} palabras del vocabulario están en GloVe ({encontradas / len(VOCAB):.0%})')

GloVe: data/glove/glove-wiki-gigaword-100.gz


19,110 de 20,002 palabras del vocabulario están en GloVe (96%)


In [14]:
def bolsa_glove(congelado):
    m = BolsaEmbeddings(d=D_G)
    with torch.no_grad():
        m.E.weight.copy_(E0)
    m.E.weight.requires_grad_(not congelado)
    return m

torch.manual_seed(0)
glove_congelado = entrenar_texto(bolsa_glove(congelado=True), epocas=5, lr=1e-2)
anotar('glove_congelado', glove_congelado)
torch.manual_seed(0)
glove_ajustado = entrenar_texto(bolsa_glove(congelado=False))
anotar('glove_ajustado', glove_ajustado)
for k, n in (('bolsa', 'E aprendido desde cero'), ('glove_congelado', 'GloVe congelado'), ('glove_ajustado', 'GloVe ajustado')):
    r = resultados[k]
    print(f"{n:24s} prueba {r['test']:.3f}")
json.dump(resultados, open('resultados_imdb_54.json', 'w'), indent=1)

época 1: exactitud de validación 0.719


época 2: exactitud de validación 0.742


época 3: exactitud de validación 0.729


época 4: exactitud de validación 0.757


época 5: exactitud de validación 0.760
7 s · 2,000,301 parámetros, 101 entrenables


glove_congelado: prueba 0.753 · prueba con palabras barajadas 0.753


época 1: exactitud de validación 0.828


época 2: exactitud de validación 0.862


época 3: exactitud de validación 0.868
9 s · 2,000,301 parámetros, 2,000,301 entrenables


glove_ajustado: prueba 0.859 · prueba con palabras barajadas 0.859
E aprendido desde cero   prueba 0.854
GloVe congelado          prueba 0.753
GloVe ajustado           prueba 0.859


In [15]:
G = nn.functional.normalize(E0, dim=1)
print('vecinos en GloVe (sin entrenar con IMDB):')
for palabra in CONSULTA:
    i = W2I[palabra]
    sim = G @ G[i]; sim[[0, 1, i]] = -1
    print(f'  {palabra:10s} →', ', '.join(VOCAB[j] for j in sim.topk(5).indices.tolist()))

vecinos en GloVe (sin entrenar con IMDB):
  excellent  → good, quality, terrific, superb, best
  terrible   → horrible, awful, dreadful, horrendous, horrific
  boring     → tedious, dull, bored, funny, stupid
  beautiful  → lovely, gorgeous, wonderful, charming, magnificent
  waste      → wastes, disposal, garbage, toxic, trash
  actor      → actress, comedian, starring, starred, actors


Dos cosas cambian.

**Los vecinos.** En GloVe, los vecinos de una palabra son palabras de **significado** parecido, y a veces de polaridad contraria: palabras que aparecen en los mismos contextos aunque digan lo opuesto. En la sección 8 los vecinos eran palabras que **empujan hacia el mismo lado**. El objetivo con que se entrenó $E$ decide qué significa estar cerca (sesión 04).

**La exactitud.** Congelado, GloVe con solo 101 parámetros entrenables queda por debajo del $E$ aprendido para la tarea: el promedio de vectores de significado no separa tan bien positivo de negativo. Ajustado, parte de un buen punto y llega a lo mismo o un poco mejor. Con 20 000 reseñas de entrenamiento hay datos de sobra para aprender $E$ desde cero; el preentrenamiento vale más cuando las etiquetas son pocas (*Tu turno*, y sesión 06).

## Tu turno · 10 minutos
1. Entrena la LSTM 6 épocas en lugar de 3. ¿Supera al promedio? ¿Y con las palabras barajadas?
2. Cambia `T_MAX` a 64 (solo el comienzo de cada reseña). ¿Qué modelo pierde más? ¿Dónde suele estar la opinión en una reseña?
3. **Pocas etiquetas.** Entrena el promedio desde cero y con GloVe congelado usando solo 200 reseñas de entrenamiento (`entrenar_texto(..., indices=i_tr[:200], epocas=30)`). ¿Cuál gana ahora?
4. Rompe la máscara a propósito: en `BolsaEmbeddings`, promedia sobre todas las posiciones, padding incluido. ¿Cambia la exactitud? ¿Más en reseñas cortas o largas?

**Para pensar.** Si para el sentimiento el promedio basta, ¿qué tarea de texto inventarías en la que un modelo sin orden fallara por construcción? (Pista: la Parte 5.)

In [16]:
# Tu turno

### Referencias
- Maas et al. (2011), *Learning Word Vectors for Sentiment Analysis*, ACL (el dataset IMDB).
- Iyyer et al. (2015), *Deep Unordered Composition Rivals Syntactic Methods for Text Classification*, ACL (promedios de embeddings).
- Kim (2014), *Convolutional Neural Networks for Sentence Classification*, EMNLP.
- Mikolov et al. (2013), *Distributed Representations of Words and Phrases and their Compositionality*, NeurIPS (word2vec).
- Pennington, Socher y Manning (2014), *GloVe: Global Vectors for Word Representation*, EMNLP.